# Analisis Data End-to-End: Faktor Penentu Harga Rumah di Jabodetabek

**Tujuan Proyek:** Menganalisis data properti hunian di wilayah Jakarta, Bogor, Depok, Tangerang, dan Bekasi (Jabodetabek) menggunakan **Python (Pandas, Matplotlib/Seaborn)**, **SQL (SQLite)**, dan **Machine Learning (Scikit-Learn)** untuk menjawab pertanyaan bisnis utama serta membangun model prediksi harga.

### Pertanyaan Bisnis Utama:
1. Bagaimana distribusi harga rumah dan harga tanah per m2 di setiap wilayah Jabodetabek?
2. Seberapa besar pengaruh jenis legalitas sertifikat (**SHM vs HGB**) terhadap valuasi properti?
3. Apakah kedekatan dengan akses tol dan stasiun transportasi (KRL/MRT/LRT) meningkatkan harga jual?
4. Fitur apa yang paling berpengaruh dalam memprediksi harga rumah menggunakan algoritma Random Forest?

In [ ]:
import json
import os
import sqlite3
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Pengumpulan & Pembersihan Data (Data Cleaning Audit)
Menjalankan pipeline pembersihan data dan memeriksa log audit keputusan pembersihan.

In [ ]:
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None

if not os.path.exists('data/clean_jabodetabek_houses.csv'):
    subprocess.run([sys.executable, 'src/01_collect_data.py'], check=True)
    subprocess.run([sys.executable, 'src/02_data_cleaning.py'], check=True)

with open('data/cleaning_audit_log.json', 'r', encoding='utf-8') as f:
    audit_log = json.load(f)

print(f"Baris awal: {audit_log['initial_rows']} | Missing awal: {audit_log['initial_missing']} -> Baris bersih: {audit_log['final_rows']}")
pd.DataFrame(audit_log['steps'])

In [ ]:
df = pd.read_csv('data/clean_jabodetabek_houses.csv')
display(df.head())
df.describe().round(2)

## 2. Analisis Bisnis Berbasis SQL (SQLite)
Menggunakan query SQL langsung ke database `data/jabodetabek_houses.db` untuk agregasi per wilayah dan sertifikat.

In [ ]:
conn = sqlite3.connect('data/jabodetabek_houses.db')

q_wilayah = '''
SELECT
    kota,
    COUNT(*) AS total_unit,
    ROUND(AVG(harga_miliar_rp), 2) AS rata_harga_miliar,
    ROUND(AVG(harga_per_m2_tanah_juta), 2) AS harga_tanah_juta_m2,
    ROUND(AVG(luas_tanah_m2), 1) AS avg_lt_m2
FROM houses
GROUP BY kota
ORDER BY harga_tanah_juta_m2 DESC;
'''
df_wilayah = pd.read_sql_query(q_wilayah, conn)
display(df_wilayah)

q_sertifikat = '''
SELECT
    sertifikat,
    COUNT(*) AS total_unit,
    ROUND(AVG(harga_miliar_rp), 2) AS rata_harga_miliar,
    ROUND(AVG(harga_per_m2_tanah_juta), 2) AS harga_tanah_juta_m2
FROM houses
GROUP BY sertifikat
ORDER BY harga_tanah_juta_m2 DESC;
'''
df_sertifikat = pd.read_sql_query(q_sertifikat, conn)
display(df_sertifikat)
conn.close()

## 3. Exploratory Data Analysis (EDA) & Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df_wilayah, x='harga_tanah_juta_m2', y='kota', hue='kota', palette='Blues_r', legend=False, ax=axes[0])
axes[0].set_title('Rata-rata Harga Tanah per m2 per Wilayah (Juta Rp)', fontweight='bold')
axes[0].set_xlabel('Juta Rp / m2')
axes[0].set_ylabel('')

sns.boxplot(data=df, x='sertifikat', y='harga_per_m2_tanah_juta', hue='sertifikat', palette='Set2', legend=False, ax=axes[1])
axes[1].set_title('Distribusi Harga Tanah per m2 Berdasarkan Sertifikat', fontweight='bold')
axes[1].set_xlabel('Jenis Sertifikat')
axes[1].set_ylabel('Juta Rp / m2')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    data=df,
    x='kategori_jarak_stasiun',
    y='harga_per_m2_tanah_juta',
    hue='akses_tol',
    order=['Sangat Dekat (<2 km)', 'Dekat (2-5 km)', 'Sedang (5-10 km)', 'Jauh (>10 km)'],
    ax=axes[0]
)
axes[0].set_title('Dampak Akses Tol & Jarak Stasiun terhadap Harga Tanah', fontweight='bold')
axes[0].set_xlabel('Jarak ke Stasiun')
axes[0].set_ylabel('Harga Tanah (Juta Rp/m2)')
axes[0].tick_params(axis='x', rotation=15)

num_cols = ['luas_tanah_m2', 'luas_bangunan_m2', 'kamar_tidur', 'kamar_mandi', 'garasi', 'jarak_stasiun_km', 'harga_miliar_rp']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('Matriks Korelasi Fitur Numerik', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Pemodelan Prediksi Harga Rumah & Feature Importance

In [ ]:
feature_cols = ['kota', 'luas_tanah_m2', 'luas_bangunan_m2', 'kamar_tidur', 'kamar_mandi', 'garasi', 'sertifikat', 'kondisi', 'akses_tol', 'jarak_stasiun_km']
X = df[feature_cols]
y = df['harga_juta_rp']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

cat_cols = ['kota', 'sertifikat', 'kondisi', 'akses_tol']
num_cols_model = ['luas_tanah_m2', 'luas_bangunan_m2', 'kamar_tidur', 'kamar_mandi', 'garasi', 'jarak_stasiun_km']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols_model),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=14, min_samples_leaf=2, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

print(f'R2 Score : {r2_score(y_test, y_pred):.4f}')
print(f'MAE      : Rp {mean_absolute_error(y_test, y_pred):.2f} Juta')
print(f'RMSE     : Rp {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} Juta')

ohe_names = rf_pipeline.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
fi_df = pd.DataFrame({'Fitur': num_cols_model + ohe_names, 'Importance': rf_pipeline.named_steps['rf'].feature_importances_})
fi_top = fi_df.sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 4))
sns.barplot(data=fi_top, x='Importance', y='Fitur', hue='Fitur', palette='viridis', legend=False)
plt.title('Top 10 Faktor Paling Berpengaruh terhadap Harga Rumah (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Kesimpulan & Rekomendasi Non-Teknis
1. **Lokasi & Luas Tanah adalah Faktor Paling Menentukan:** Kombinasi luas tanah, luas bangunan, dan lokasi administratif (khususnya Jakarta Selatan vs kota penyangga) menjelaskan mayoritas variasi harga pasar.
2. **Sertifikat SHM Memberikan Premi Harga:** Konsumen bersedia membayar lebih untuk kepastian legalitas SHM.
3. **Akses Infrastruktur Meningkatkan Nilai Investasi:** Kedekatan dengan pintu tol dan stasiun KRL/MRT/LRT (<2 km) terbukti menaikkan valuasi harga tanah secara konsisten.